# W&B Experiment Tracking — Live Training of a TinyTransolver

This notebook trains a **~896k-parameter TinyTransolver** — a small,
workshop-friendly surrogate inspired by the Transolver architecture
used in industrial CFD workflows — on the synthetic NACA airfoil
dataset that `registry.ipynb` promoted into the W&B Registry. Along the
way you will see:

1. A training loop that logs loss metrics to W&B in real time.
2. Per-epoch model checkpoints saved as versioned model artifacts.
3. The best-epoch checkpoint linked into the `airfoil_surrogates`
   Registry under the `checkpoints` Collection.
4. W&B Tables, custom charts, and Transolver-specific custom metrics —
   logged inline as training happens.
5. A small pointwise MLP baseline so the audience can see what the
   Physics-Attention blocks buy over independent per-point regression.
6. A curated four-section W&B Workspace view saved after training so
   the live demo opens directly on the important model media.

The training run is opened inside a `with wandb.init(...) as run:`
block so it finishes cleanly on exit.

**Prerequisites**
- `uv sync` (installs `wandb`, `torch`, `numpy`, `matplotlib` per `pyproject.toml`).
- `uv run wandb login`.
- **`registry.ipynb` executed first** — it produces
  `wandb-registry-airfoil_surrogates/datasets:production`, which this
  notebook consumes at its setup step.
- Access to the `wandb_emea` entity on W&B, or edit the `ENTITY`
  constant in §1 below.

**Background reading**
- Physics-Attention and the base Transolver: `docs/explainers/pxtransolver.md`.
- Source paper: Wu et al. (2024), *Transolver: A Fast Transformer Solver for PDEs on General Geometries*, arXiv:2402.02366.


## 1. Setup

Third-party imports plus shared workshop helpers from `src/physicsx_demo/utils`.
The notebook stays focused on the W&B experiment flow; reusable data,
model, metric, plotting, and training code lives in the package.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json
import math
import tempfile

import numpy as np
import torch
import torch.nn.functional as F
import wandb

from physicsx_demo.utils.env import configure_matplotlib, load_local_env, resolve_wandb_target

configure_matplotlib()

import matplotlib.pyplot as plt

from physicsx_demo.utils.data import cases_from_payload, split_cases, stack_cases
from physicsx_demo.utils.metrics import proxy_metrics, relative_l2, selection_score, slice_regularization
from physicsx_demo.utils.model import PointwiseMLP, TinyTransolver, count_params
from physicsx_demo.utils.viz import (
    airfoil_gallery_figure,
    airfoil_profile_figure,
    comparison_figure,
    error_figure,
    save_figure_png,
    save_frame_png,
    scorecard_figure,
    slice_figure,
    slice_usage_figure,
    slice_video_frame,
    write_slice_evolution_gif,
)

load_local_env()

device = torch.device("cpu")
ENTITY, PROJECT = resolve_wandb_target()

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"target = {ENTITY}/{PROJECT}")
print(f"device = {device}")


In [ ]:
# AirfoilCase and JSON loading helpers are imported from `physicsx_demo.utils.data`.
# The model sees each case through AirfoilCase.features() and AirfoilCase.targets().


## 2. Data — from the W&B Registry

The synthetic NACA airfoil dataset was generated and promoted to the
Registry by `registry.ipynb`. This notebook consumes it via
`wandb.Api().artifact("wandb-registry-airfoil_surrogates/datasets:production")`
— the same path any downstream consumer (sweeps, reports, ...) uses.

Each `AirfoilCase` is a point-cloud snapshot of a NACA-4-digit airfoil
under one `(angle_of_attack, Reynolds_number)`. Features per point:
`(x, y, sdf, aoa, log_re)`. Targets per point: `(pressure, u, v)`.
Payload is JSON so it's easy to inspect.


In [ ]:
# ---- Consume the dataset from the W&B Registry ----
REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
VAL_FRACTION = 0.2

api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
art = api.artifact(DATASET_REGISTRY_PATH)
ds_dir = Path(art.download())
json_file = next(ds_dir.glob("*.json"))
with open(json_file) as f:
    payload = json.load(f)

cases = cases_from_payload(payload)
train_cases, val_cases = split_cases(cases, VAL_FRACTION)

SEED = int(payload.get("meta", {}).get("seed", 42))
N_POINTS = int(payload.get("meta", {}).get("n_points", 512))

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"pulled {DATASET_REGISTRY_PATH}")
print(f"dataset meta: {payload['meta']}")
print(f"total cases: {len(cases)}   train: {len(train_cases)}   val: {len(val_cases)}")
print(f"seed: {SEED}")
print(f"n_points per case: {N_POINTS}")

sample = train_cases[0]
print()
print("One AirfoilCase (what the model sees):")
print(f"  features shape : {sample.features().shape}   (per point: [x, y, sdf, aoa, log_re])")
print(f"  targets  shape : {sample.targets().shape}    (per point: [pressure, u, v])")
print(f"  angle of attack: {math.degrees(sample.aoa):.2f} deg")
print(f"  Reynolds number: {sample.re:.1f}")


## 3. The model — a tiny (~896k-parameter) Transolver

The model is defined inline so the Physics-Attention mechanism is easy
to read during the workshop. It keeps the three core Transolver moves:

1. **Slice** each point into `M = 64` learned physics slices.
2. **Attend** between the small set of slice tokens, not every pair of
   mesh points.
3. **De-slice** the updated slice tokens back to the original point cloud.
4. **Pool a global context token** from the full point cloud and broadcast it back before prediction.

This gives the audience the right mental model for the paper's
`O(NM + M^2)` path without requiring a full industrial mesh or GPU-scale
training run. The notebook also trains a pointwise MLP baseline with no
cross-point communication, which makes the value of slice-level and
global message passing visible in W&B.


In [ ]:
# PhysicsAttention and TinyTransolver are imported from `physicsx_demo.utils.model`.


In [ ]:
# PointwiseMLP and count_params are imported from `physicsx_demo.utils.model`.


In [ ]:
# ---- Instantiate and sanity-check ----
MODEL_CONFIG = {
    "in_dim": 5,
    "out_dim": 3,
    "hidden": 128,
    "n_layers": 6,
    "n_slices": 64,
    "n_heads": 8,
    "ffn_expansion": 2,
    "dropout": 0.05,
    "use_global_context": True,
}

model = TinyTransolver(**MODEL_CONFIG).to(device)
baseline = PointwiseMLP(in_dim=MODEL_CONFIG["in_dim"], out_dim=MODEL_CONFIG["out_dim"]).to(device)
print(model)
print()
print(f"TinyTransolver parameters: {count_params(model):,}")
print(f"Pointwise MLP parameters : {count_params(baseline):,}")

with torch.no_grad():
    dummy_features = torch.from_numpy(train_cases[0].features()).float().unsqueeze(0)  # (1, N, 5)
    dummy_out = model(dummy_features)
    dummy_base = baseline(dummy_features)
    print(f"TinyTransolver forward: in={tuple(dummy_features.shape)} -> out={tuple(dummy_out.shape)}")
    print(f"Pointwise MLP forward : in={tuple(dummy_features.shape)} -> out={tuple(dummy_base.shape)}")


## 4. Train the model and promote the best checkpoint

One W&B run covers the whole training session: consume the dataset
from the Registry (creating a lineage edge), train the Transolver,
train a pointwise MLP baseline for context, log per-epoch metrics +
checkpoints + visualisations, ensure the custom Registry exists, and
link the best-val checkpoint into it. Everything is inside one
`with wandb.init(job_type="train") as run:` block.

### Audit: where this physical example runs short

This is intentionally a **surrogate-modeling workshop example**, not a
CFD benchmark. Compared with the Transolver paper, it is short in four
important ways:

- The dataset is a small analytic proxy, not a Navier-Stokes/RANS CFD
  dataset. Reynolds number is present as a condition, but the generator
  only uses it weakly, so the model cannot learn rich Reynolds-regime
  effects.
- The points are a lightweight cloud around an airfoil, not a
  high-resolution unstructured mesh with boundary layers, surface
  normals, connectivity, or wall-shear information.
- The targets are pressure and velocity fields, but there is no true
  lift/drag integral from CFD. We log a clearly labelled pressure-force
  **proxy** so the workshop can still show design-ranking metrics.
- The TinyTransolver shares the same simplified code path as the
  workshop notebooks. It keeps the slice-attend-deslice idea, but it is
  much smaller and less tuned than the paper's A100-scale experiments.

The improvements below make those limitations visible instead of
hiding them: we log relative L2 field error, pressure-force proxy MAE,
Spearman rank correlation for design ranking, slice-health metrics, a
composite model-selection score, and an MLP baseline guardrail.

**Data preparation before the loop.** We compute per-channel `mean` and
`std` on the **training split only** and standard-normalise both
features and targets. Training MSE is measured in std-units, so
pressure and velocity contribute comparably to the loss. Predictions
are denormalised when we render pressure-field plots and compute
physical-space proxy metrics.

**Loss.** Two terms.

- **`mse`** — mean-squared error on the normalised targets.
- **`slice_sharpness`** — normalised per-point entropy. Lower means
  each point commits to fewer slices.
- **`slice_usage`** — normalised KL from uniform slice usage. Lower
  means all slices stay alive across the batch.

`total_loss = mse + lambda_slice_sharpness * slice_sharpness + lambda_slice_usage * slice_usage`.

**Per-epoch work.**

1. Train TinyTransolver and the pointwise MLP baseline on the same batches.
2. Log field metrics: normalised RMSEs and physical-space relative L2.
3. Log design metrics: lift/drag proxy MAE and lift rank Spearman.
4. Log slice metrics: point entropy, usage entropy, and effective slice count.
5. Save a versioned model artifact every epoch, select the best checkpoint by composite score, and link it to W&B Models/Registry. If the pointwise baseline wins the same score, log a guardrail warning and save/link the baseline artifact too.


In [ ]:
# Plotting, media helpers, and workshop metrics are imported from
# `physicsx_demo.utils.viz` and `physicsx_demo.utils.metrics`.


In [ ]:
# ---- Train, log rich W&B media, and promote the best checkpoint ----
import importlib
import physicsx_demo.utils.viz as viz_utils

# Reload plotting helpers before importing training; existing notebook
# kernels can otherwise keep a stale viz module without slice_usage_frame.
viz_utils = importlib.reload(viz_utils)
import physicsx_demo.utils.training as training_utils

# Reload during workshop iteration so an already-running notebook kernel
# picks up fixes in src/physicsx_demo/utils/training.py.
training_utils = importlib.reload(training_utils)
run_training_experiment = training_utils.run_training_experiment

TRAIN_CONFIG = run_training_experiment(
    entity=ENTITY,
    project=PROJECT,
    registry_name=REGISTRY_NAME,
    dataset_registry_path=DATASET_REGISTRY_PATH,
    model_config=MODEL_CONFIG,
    model=model,
    baseline=baseline,
    train_cases=train_cases,
    val_cases=val_cases,
    n_points=N_POINTS,
    seed=SEED,
    device=device,
)


## 5. Curate the W&B Workspace

The training run has already logged the media. This cell only controls
how the live W&B Workspace presents it during the workshop. We disable
auto-generated panels and create exactly four model media sections:

- `model/slice_usage_evolution_video`
- `model/slice_usage_epoch_n`
- `model/slice_assignment_evolution_video`
- `model/slice_assignment_epoch_n`

This avoids the default auto-grouped `model` section and gives the
audience a cleaner cockpit immediately after training finishes.


In [ ]:
import wandb_workspaces.reports.v2 as wr
import wandb_workspaces.workspaces as ws

MEDIA_EPOCHS = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 31]
slice_usage_epoch_keys = [f"model/slice_usage_epoch_{epoch}" for epoch in MEDIA_EPOCHS]
slice_assignment_epoch_keys = [f"model/slice_assignment_epoch_{epoch}" for epoch in MEDIA_EPOCHS]

workspace = ws.Workspace(
    entity=ENTITY,
    project=PROJECT,
    name="PhysicsX Workshop - Models",
    settings=ws.WorkspaceSettings(
        x_axis="epoch",
        smoothing_type="exponential",
        smoothing_weight=20,
        max_runs=12,
    ),
    runset_settings=ws.RunsetSettings(
        query="tiny-transolver",
        pinned_columns=[
            "State",
            "Created",
            "summary.model/best_selection_score",
            "summary.baseline/best_selection_score",
            "summary.model/baseline_guardrail_passed",
        ],
    ),
    auto_generate_panels=False,
    sections=[
        ws.Section(
            name="model/slice_usage_evolution_video",
            is_open=True,
            panels=[
                wr.MediaBrowser(
                    title="model/slice_usage_evolution_video",
                    media_keys=["model/slice_usage_evolution_video"],
                    num_columns=1,
                    mode="gallery",
                ),
            ],
        ),
        ws.Section(
            name="model/slice_usage_epoch_n",
            is_open=True,
            panels=[
                wr.MediaBrowser(
                    title="model/slice_usage_epoch_n",
                    media_keys=slice_usage_epoch_keys,
                    num_columns=3,
                    mode="gallery",
                ),
            ],
        ),
        ws.Section(
            name="model/slice_assignment_evolution_video",
            is_open=True,
            panels=[
                wr.MediaBrowser(
                    title="model/slice_assignment_evolution_video",
                    media_keys=["model/slice_assignment_evolution_video"],
                    num_columns=1,
                    mode="gallery",
                ),
            ],
        ),
        ws.Section(
            name="model/slice_assignment_epoch_n",
            is_open=True,
            panels=[
                wr.MediaBrowser(
                    title="model/slice_assignment_epoch_n",
                    media_keys=slice_assignment_epoch_keys,
                    num_columns=3,
                    mode="gallery",
                ),
            ],
        ),
    ],
)

workspace.save()
WORKSPACE_URL = workspace.url
print(f"curated workspace: {WORKSPACE_URL}")


## 6. Wrap up — what to look at in the W&B UI

One run was created by this notebook (`tiny-transolver-train`,
`job_type = train`). In the W&B UI you should see:

**Lineage graph**
- `wandb-registry-airfoil_surrogates/datasets:production` (produced
  by `registry.ipynb`) → this training run → newly-linked
  `wandb-registry-airfoil_surrogates/checkpoints:production`.

**Training run metrics**
- Line charts for `train/loss`, `train/weighted_field_mse`, `val/weighted_mse`, `val/pressure_rmse`,
  `val/field_relative_l2`, `model/selection_score`, and `model/selection_margin_vs_baseline`.
- Baseline context under `baseline/*` so the audience can compare
  Physics-Attention against independent per-point regression.
- Design-proxy metrics: `val/lift_proxy_mae`, `val/drag_proxy_mae`,
  `val/lift_proxy_spearman`, and `val/drag_proxy_spearman`.
- Slice-health metrics: `slice/point_entropy`, `slice/usage_entropy`,
  and `slice/effective_slices`.

**Artifacts logged by the training run**
- `airfoil_surrogate_ckpt:v0..v31` — one model artifact per epoch,
  aliased `epoch-N`, each with checkpoint metadata for field error,
  ranking proxy, and baseline comparison.

**Media panels and Tables**
- `dataset/canonical_airfoil_profile`, `workshop/airfoil_profile_reference`, `workshop/airfoil_profile_gallery`, and `workshop/airfoil_profiles_table` are easy-to-find run media for the physical airfoil shape. They are logged once because geometry is a fixed dataset input, not an epoch-evolving model output.
- Artifact-backed image Tables are also logged as `workshop_airfoil_profiles:latest`, `workshop_case_diagnostics:latest`, `workshop_slice_usage_frames:latest`, and `workshop_slice_assignment_frames:latest` so image cells remain inspectable when downloaded or viewed from Artifacts.
- `summary/scorecard_epoch_*` standalone `wandb.Image` summaries for model score, baseline score, guardrail margin, and slice health.
- `dataset/canonical_airfoil_profile` and `workshop/airfoil_profile_reference` static geometry images for the canonical validation case.
- `val/comparison_epoch_*` three-panel pressure images: truth vs TinyTransolver vs baseline.
- `val/error_map_epoch_*` pressure-error heatmaps for model and baseline.
- `model/slice_usage_epoch_*` and `model/slice_assignment_epoch_*` routing images.
- `model/slice_usage_evolution_video` and `model/slice_assignment_evolution_video` animated GIFs logged with `wandb.Video`, plus matching per-epoch frame Tables.
- `val/case_diagnostics_epoch_*` rich `wandb.Table`s with per-case metrics plus airfoil profile, comparison, error, and slice images.
- `val/case_error_leaderboard_epoch_*` and `val/rmse_by_aoa_epoch_*` bar charts.

**Curated Workspace**
- The final code cell saves `PhysicsX Workshop - Models` with exactly four sections: `model/slice_usage_evolution_video`, `model/slice_usage_epoch_n`, `model/slice_assignment_evolution_video`, and `model/slice_assignment_epoch_n`.
- Use the printed `WORKSPACE_URL` rather than the default auto-generated project workspace when presenting.

**W&B Models / Registry**
- Registry `airfoil_surrogates` has the core Collections used by the workshop:
  - `datasets` (from `registry.ipynb`).
  - `checkpoints` — best TinyTransolver version aliased `production` and
    `epoch-N`.
  - `baselines` — created only when the pointwise guardrail beats the
    TinyTransolver composite score.

Run `sweeps.ipynb` next to fine-tune the champion checkpoint.
